In [ ]:
# !pip install torch transformers datasets sentencepiece tqdm scipy sentence-transformers datasets evaluate scikit-learn evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer, models, losses, InputExample, evaluation
import torch
import os
from datasets import load_dataset
import numpy as np
from typing import Tuple
from torch import nn
from datasets import load_dataset, DatasetDict, load_from_disk  # Import load_from_disk
from sentence_transformers.readers import STSBenchmarkDataReader
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
import evaluate
import pandas as pd

In [ ]:
parquet_file_path = "/content/songs.parquet"

try:
    df_songs = pd.read_parquet(parquet_file_path, engine="pyarrow")  # or engine='fastparquet'
    print("Parquet file loaded successfully!")
    display(df_songs.head())
except FileNotFoundError:
    print(f"Error: The file {parquet_file_path} was not found.")
except Exception as e:
    print(f"An error occurred while loading the parquet file: {e}")

Parquet file loaded successfully!


,corpus,doc_id,sent_index,sentence,split,token_count
0,taylor_songs,1989_Booklet_,0,"The drought was the very worst (Oh-oh, oh-oh)",train,8
1,taylor_songs,1989_Booklet_,1,When the flowers that we'd grown together died...,train,10
2,taylor_songs,1989_Booklet_,2,It was months and months of back and forth (Oh...,train,11
3,taylor_songs,1989_Booklet_,3,You're still all over me,train,5
4,taylor_songs,1989_Booklet_,4,Like a wine-stained dress I can't wear anymore,train,8


In [ ]:
model = SentenceTransformer("bert-base-uncased")
print("Model 'bert-base-uncased' loaded successfully.")

Model 'bert-base-uncased' loaded successfully.


In [ ]:
sentences = df_songs["sentence"][:100]

sentence_embeddings = model.encode(
    sentences,
)

print("Sentence embeddings generated.")
print(f"Number of sentences: {len(sentences)}")
print(f"Shape of embeddings: {sentence_embeddings.shape}")

# Display the first embedding as an example
print("\nFirst sentence embedding (first 5 dimensions):")
print(sentence_embeddings[0][:5])

Sentence embeddings generated.
Number of sentences: 100
Shape of embeddings: (100, 768)

First sentence embedding (first 5 dimensions):
[-0.12906848  0.346674    0.5359276  -0.24112368  0.00692912]


In [ ]:
output_dim = 32

pca = PCA(n_components=output_dim)
sentence_embeddings_pca = pca.fit_transform(sentence_embeddings)

print("Original sentence embeddings shape:", sentence_embeddings.shape)
print("PCA reduced sentence embeddings shape:", sentence_embeddings_pca.shape)

Original sentence embeddings shape: (100, 768)
PCA reduced sentence embeddings shape: (100, 32)
